<p style="text-align:center"><img src="../images/04_00_spatial_stat.png" alt="Spatial Statistics" width="100%"></p>


# Core Theoretical Concepts

This notebook is Foundation Chapter 0.2 of 3. It develops the vocabulary that every later Part assumes: Tobler's First Law and spatial dependence, the three spatial data types, stationarity, the Modifiable Areal Unit Problem, Complete Spatial Randomness as a null baseline, and spatial weights matrices.

| Chapter | Notebook | Focus |
|---|---|---|
| **0.1** | [`04_00_01_spatial_statistics_introduction.ipynb`](04_00_01_spatial_statistics_introduction.ipynb) | What spatial statistics is and why it matters |
| **0.2** (this) | [`04_00_02_core_theoretical_concepts.ipynb`](04_00_02_core_theoretical_concepts.ipynb) | Dependence, support, stationarity, MAUP, CSR, weights |
| **0.3** | [`04_00_03_module_map_and_learning_paths.ipynb`](04_00_03_module_map_and_learning_paths.ipynb) | Package map, Part overview, setup, learning paths |


## Overview

| Concept | One-line definition | Where it reappears |
|---|---|---|
| Spatial dependence | Nearby observations are more similar than distant ones | Parts 2–6, 9 |
| Support (data type) | Geometric form of each observation: point, area, or field | Method choice in every Part |
| Stationarity | Statistical properties do not change with location | Kriging, global models, GWR contrast |
| MAUP | Results change with scale and zonation of aggregation | Parts 1–2, 5–6 |
| CSR | Homogeneous Poisson process as the point-pattern null | Part 1 |
| Spatial weights $W$ | Encoded neighbour relationships for lattice data | Parts 2–6, 9 |

A short setup cell at the end confirms that `geolattice.spatial_statistics` is importable; the full environment walkthrough is in Chapter 0.3.


## Tobler's First Law and Spatial Dependence

> *"Everything is related to everything else, but near things are more related than distant things."*  
> — Waldo Tobler (1970)

**Spatial dependence** (spatial autocorrelation) is the tendency for geographically proximate observations to be more similar than those far apart. It is not a nuisance to be corrected away; it is information about the underlying process.

### Positive vs. negative autocorrelation

| Pattern | Description | Typical domains |
|---|---|---|
| **Positive** | Like clusters with like; unlike with unlike | Socio-economic rates, climate fields, disease prevalence |
| **Negative** | Checkerboard / competitive arrangement | Retail location under fierce competition; some predator–prey territories |

Positive autocorrelation dominates applied spatial data. Negative autocorrelation is rarer and usually signals an explicit competitive or inhibitory mechanism.

### Formal definition

For a random variable $Z(s)$ observed at location $s$, spatial dependence is captured by the spatial covariance function:

$$C(s_i, s_j) = \operatorname{Cov}\bigl[Z(s_i),\, Z(s_j)\bigr]$$

Under **second-order stationarity**, $C$ depends only on the separation vector $h = s_i - s_j$, not on absolute location. Then we write $C(h)$ and speak of a *covariance function* or, equivalently, of a *variogram*

$$\gamma(h) = \tfrac{1}{2}\operatorname{Var}\bigl[Z(s) - Z(s+h)\bigr].$$

Part 7 (interpolation) estimates $\gamma(h)$ from data and feeds it to kriging. Parts 2–3 encode the same idea more discretely through a weights matrix $W$.

### Dependence vs. heterogeneity

Two distinct phenomena create the *appearance* of spatial pattern:

- **Dependence**: interaction or shared unmeasured factors make neighbouring values correlated.
- **Heterogeneity**: the mean, variance, or covariate effects themselves change across space (non-stationarity).

Distinguishing them is one of the hardest inferential problems in the field. Global Moran statistics detect structure; geographically weighted models (Part 6) and hierarchical random effects (Part 5) help attribute that structure to local parameter variation vs. residual correlation.


## The Three Spatial Data Types

The choice of method depends entirely on the **support**—the geometric form—of the data.

| Type | Description | Example | Analysis family |
|---|---|---|---|
| **Point pattern** | A set of event locations in a study window; the *locations themselves* are the data | Crime incidents, disease cases, tree positions | Point Pattern Analysis (Part 1) |
| **Lattice / areal** | A finite set of polygons, each with an associated attribute | County-level diabetes prevalence | Autocorrelation, regression, clustering (Parts 2–5) |
| **Geostatistical field** | A continuous surface sampled at finite locations; the surface extends everywhere | Air quality at monitors; elevation | Interpolation / kriging (Part 7) |

### Matching support to probabilistic model

| Support | Probabilistic model | Key object |
|---|---|---|
| Point pattern | Spatial point process $\Phi$ | Intensity $\lambda(s)$; second-order functions $K$, $L$, … |
| Lattice / areal | Gaussian Markov random field / spatial weights | Matrix $W$; Moran $I$; lag/error models |
| Continuous field | Random function / regionalised variable | Variogram $\gamma(h)$; kriging predictor |

### Decision guide

Ask three questions before choosing a Part:

1. **Are the coordinates the data, or attributes attached to fixed units?** If coordinates *are* the data → Part 1. If attributes on fixed polygons → Parts 2–6. If attributes on sample points of a continuous surface → Part 7.
2. **Is the study region a window that could contain events anywhere, or a tessellation of areas?** Windows → point processes. Tessellations → lattice methods.
3. **Do you need a value *between* sample locations?** Yes → interpolation. No → stick with the observed support.

Mis-matching support is the most common beginner error: aggregating points to counties and then running a point-process $K$-function, or kriging a lattice variable as if it were a continuous field, both produce answers that look numeric and are scientifically wrong.


## Stationarity and Non-Stationarity

**Stationarity** means that the statistical properties of a spatial process do not change with location.

### Levels of stationarity

| Level | Meaning | Where it is used |
|---|---|---|
| **Strict** | Full joint distribution is shift-invariant | Rarely verifiable; theoretical baseline |
| **Second-order (weak)** | Constant mean; covariance depends only on lag $h$ | Most global spatial models |
| **Intrinsic** | Only the *variogram of differences* is stationary | Ordinary kriging |

### Sources of non-stationarity

**Non-stationarity** (spatial heterogeneity) means means, variances, or covariances vary across space. Common sources:

- **Macro-scale trend (drift)** — addressed by trend removal or universal kriging.
- **Regime changes** — addressed by locally weighted models (GWR, Part 6).
- **Varying error structure** — addressed by heteroskedastic or spatial error models.

Global models (ordinary Moran $I$, SAR with a single $\rho$, ordinary kriging) assume some form of stationarity. Local models (LISA, GWR, MGWR) exist because that assumption often fails. Always ask: *is a single global parameter scientifically plausible here?*


## The Modifiable Areal Unit Problem (MAUP)

The **MAUP** is the sensitivity of statistical results to the choice of spatial unit used to aggregate data. It has two dimensions:

| Dimension | Description | Typical symptom |
|---|---|---|
| **Scale effect** | Aggregating to larger units smooths within-unit variability | Inflated correlations at coarser scales |
| **Zonation effect** | Equally valid partitions at the same scale yield different results | Coefficients that flip when boundaries are redrawn |

### Why there is no universal fix

The MAUP is a property of *aggregation*, not a bug in a particular software package. Practical mitigations:

1. **Report multi-scale results** — recompute key statistics at tract, county, and regional levels.
2. **Prefer the support that matches the process** — do not aggregate points to counties solely for convenience if the process is point-level.
3. **Let the data choose scale where possible** — GWR/MGWR bandwidths (Part 6) and Bayesian hierarchical models (Part 5) partially absorb scale choice into the model.
4. **Document zonation** — when using administrative boundaries, say so; they are political artefacts, not natural process units.

Part 1 revisits MAUP at the point-to-polygon boundary; Part 2 treats weight-choice and boundary sensitivity as first-class exercises.


## Complete Spatial Randomness (CSR)

In point pattern analysis, the reference process is the **Homogeneous Poisson Process** (HPP), also called Complete Spatial Randomness:

1. The number of events in any region $A$ follows $\operatorname{Poisson}(\lambda |A|)$ where $\lambda$ is a constant intensity.
2. Event locations are **independent** of each other.

### What departures mean

| Departure | Interpretation | Examples |
|---|---|---|
| **Clustering** | Attraction, contagion, or shared covariates | Crime near nightlife; disease around a source |
| **Regularity / inhibition** | Competition or territorial exclusion | Tree spacing under competition; store exclusion zones |

### Why CSR is the right null

Classical nulls ("mean difference is zero") do not encode geography. CSR does: it says *events are placed uniformly and independently in the window*. Every summary statistic in Part 1—intensity functions, $K$, $L$, $G$, $F$, $J$—is designed to quantify deviations from that baseline, usually with Monte-Carlo envelopes under simulated CSR (or under an inhomogeneous Poisson null when intensity varies).

Under CSR, Ripley's $K$-function satisfies $K(r) = \pi r^2$. Values above signal clustering; values below signal regularity.


## Spatial Weights

For lattice data, spatial dependence is encoded in a **spatial weights matrix** $W$—an $n \times n$ matrix where $w_{ij}$ encodes the strength of the connection between areas $i$ and $j$.

### Common schemes

| Scheme | Rule | $w_{ij}$ |
|---|---|---|
| **Rook contiguity** | Share an edge | 1 if touching edge, else 0 |
| **Queen contiguity** | Share an edge or vertex | 1 if touching corner or edge, else 0 |
| **$k$-nearest neighbour (KNN)** | $k$ closest centroids | 1 for $k$ nearest, else 0 |
| **Distance band** | Within threshold $d$ | 1 if $d(i,j) < d$, else 0 |
| **Inverse-distance** | Continuous decay | $1 / d(i,j)^{\alpha}$ |

### Row-standardisation

Weights are typically **row-standardised** so $\sum_j w_{ij} = 1$. Then the spatial lag $Wy$ is a local average of neighbours' values—interpretable and numerically stable for Moran $I$ and spatial regression.

### Modelling consequences

The choice of $W$ is a modelling decision, not a preprocessing detail:

- Contiguity suits compact administrative lattices (counties, tracts).
- KNN suits irregular lattices or islands where contiguity leaves isolates.
- Distance bands require a scientifically motivated $d$ (commute shed, dispersal distance).
- Higher-order weights ($W^2$, $W^3$) capture multi-step neighbourhoods but inflate density.

All downstream statistics—Moran's $I$, LISA, spatial lag/error models, BYM CAR priors—inherit $W$. Part 2 treats weight construction and sensitivity as a full chapter; Parts 3–6 reuse the same `geolattice.spatial_statistics.weights` objects.


## Putting the Concepts Together

A compact checklist before any Module 04 analysis:

1. **Name the support** — point pattern, lattice, or field.
2. **State the scientific question** — pattern, dependence, process, cluster, map, heterogeneity, prediction, design, or dynamics.
3. **Choose the null** — CSR for points; permutation of values (or residual Moran) for lattices; variogram under a chosen stationarity level for fields.
4. **Declare scale and zonation** — especially if you aggregated.
5. **Document $W$** — scheme, standardisation, and isolates—whenever lattice methods are used.
6. **Separate dependence from heterogeneity** — do not interpret a significant Moran $I$ as proof that GWR is required, or vice versa.

---

## Setup check

Confirm that the repository root resolves and `geolattice.spatial_statistics` imports. Full submodule coverage and learning paths follow in Chapter 0.3.

**Installation.** This notebook needs installation option 3 (core, spatial analysis and spatial statistics): `pip install "geolattice[statistics]"`. See "Installation options" in the GeoLattice README.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").is_file() and (candidate / "Data").is_dir():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find the GeoLattice repository root.")
sys.path.insert(0, str(ROOT))

import geolattice
import geolattice.spatial_statistics as sstat

print(f"geolattice: {geolattice.__version__}")
print(f"geolattice.spatial_statistics: {sstat.__version__}")
print("Core concepts notebook environment: OK")


geolattice: 0.3.49
geolattice.spatial_statistics: 0.3.49
Core concepts notebook environment: OK


**Next:** Open [`04_00_03_module_map_and_learning_paths.ipynb`](04_00_03_module_map_and_learning_paths.ipynb) for the `geolattice.spatial_statistics` architecture, a Part-by-Part notebook map, setup, and suggested learning paths.


## Summary and Conclusion